# BirdCLEF 2026 — Pseudo-Label Full Inference (Round 2)

Runs the V3+V9 ensemble (ConvNeXt-Small v3 + ECA-NFNet-L0 v9) on all 10,592 unlabeled
train soundscapes and saves per-clip predictions as a parquet file.

Round 1 used V5+V9 (ensemble score 0.803). Round 2 upgrades Model A to V3
(pseudo-label-trained ConvNeXt-Small, 0.805 solo, 0.811 ensemble) for higher-quality labels.

Each row = one 5-second clip (at 2.5-second stride) from one soundscape.
Columns: `filename`, `clip_index`, `clip_start_sec`, then 234 class probabilities.

Estimated runtime: ~4.5 hours on T4 GPU.

In [ ]:
import warnings
from pathlib import Path

import librosa
import numpy as np
import pandas as pd
from PIL import Image
import torch
from torchvision import transforms as T
from fastai.vision.all import load_learner

warnings.filterwarnings('ignore', category=UserWarning, module='fastai')

In [ ]:
import kagglehub
competition_dir = Path(kagglehub.competition_download('birdclef-2026'))
print('Competition dir:', competition_dir)

labels_df = pd.read_csv(competition_dir / 'train_soundscapes_labels.csv')
labeled_files = set(labels_df['filename'].unique())
print(f'Labeled soundscape files (excluded from pseudo-labeling): {len(labeled_files)}')

all_soundscapes = sorted((competition_dir / 'train_soundscapes').glob('*.ogg'))
unlabeled = [f for f in all_soundscapes if f.name not in labeled_files]
print(f'Total train soundscapes: {len(all_soundscapes)}')
print(f'Unlabeled (pseudo-label targets): {len(unlabeled)}')

In [ ]:
MODEL_A_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234_v3/2/model_multilabel_234.pkl'
MODEL_B_PATH = '/kaggle/input/models/ucheozoemena/bird-clef-classifier/pytorch/multilabel_234/9/model_multilabel_234.pkl'

HOP_LENGTH      = 512
SAMPLE_RATE     = 32000
CLIP_DURATION   = 5
STRIDE_DURATION = 2.5
TARGET_SIZE     = (224, 224)
BATCH_SIZE      = 64
LOG_EVERY       = 500   # print progress every N files

In [ ]:
learn_a = load_learner(MODEL_A_PATH)
learn_b = load_learner(MODEL_B_PATH)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learn_a.model = learn_a.model.to(device).eval()
learn_b.model = learn_b.model.to(device).eval()
print(f'Running on: {device}')

vocab = list(learn_a.dls.vocab)
print(f'Classes: {len(vocab)}')

In [ ]:
clip_length    = int(CLIP_DURATION * SAMPLE_RATE)
stride_samples = int(STRIDE_DURATION * SAMPLE_RATE)
stride_frames  = int(STRIDE_DURATION * SAMPLE_RATE / HOP_LENGTH)
frames_per_clip = int(CLIP_DURATION * SAMPLE_RATE / HOP_LENGTH)

tfm = T.Compose([
    T.Resize(TARGET_SIZE),
    T.ToTensor(),
    T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

def window_to_img(window):
    lo, hi = float(window.min()), float(window.max())
    arr = np.zeros_like(window, dtype=np.uint8) if hi == lo else (
        (window - lo) / (hi - lo) * 255
    ).astype(np.uint8)
    return Image.fromarray(arr).resize(TARGET_SIZE).convert('RGB')

def build_clips(S_db, n_samples):
    imgs, starts = [], []
    k = 0
    while k * stride_samples + clip_length <= n_samples:
        window = S_db[:, k * stride_frames : k * stride_frames + frames_per_clip]
        imgs.append(window_to_img(window))
        starts.append(k * STRIDE_DURATION)
        k += 1
    return imgs, starts

def run_inference(learn, imgs):
    preds = []
    for i in range(0, len(imgs), BATCH_SIZE):
        batch = torch.stack([tfm(img) for img in imgs[i:i+BATCH_SIZE]]).to(device)
        with torch.no_grad():
            logits = learn.model(batch)
        preds.append(torch.sigmoid(logits).cpu().numpy())
    return np.vstack(preds)

In [ ]:
all_preds      = []   # list of (n_clips, 234) float32 arrays
all_filenames  = []   # one entry per clip
all_clip_idx   = []   # clip index within soundscape
all_clip_start = []   # clip start time in seconds

n_total = len(unlabeled)

for i, soundscape in enumerate(unlabeled):
    samples, _ = librosa.load(soundscape, sr=SAMPLE_RATE)

    S_db = librosa.power_to_db(
        librosa.feature.melspectrogram(y=samples, sr=SAMPLE_RATE, hop_length=HOP_LENGTH),
        ref=np.max,
    )

    imgs, starts = build_clips(S_db, len(samples))
    if not imgs:
        continue

    preds_a = run_inference(learn_a, imgs)
    preds_b = run_inference(learn_b, imgs)
    ensemble = ((preds_a + preds_b) / 2.0).astype(np.float32)

    all_preds.append(ensemble)
    n_clips = len(imgs)
    all_filenames.extend([soundscape.name] * n_clips)
    all_clip_idx.extend(range(n_clips))
    all_clip_start.extend(starts)

    if (i + 1) % LOG_EVERY == 0 or (i + 1) == n_total:
        total_clips = sum(len(p) for p in all_preds)
        print(f'[{i+1:>5}/{n_total}] {soundscape.name} — {total_clips:,} clips accumulated')

print(f'\nInference complete. Total clips: {sum(len(p) for p in all_preds):,}')

In [ ]:
preds_np = np.vstack(all_preds)  # (total_clips, 234)

df = pd.DataFrame(preds_np, columns=vocab)
df.insert(0, 'clip_start_sec', all_clip_start)
df.insert(0, 'clip_index', all_clip_idx)
df.insert(0, 'filename', all_filenames)

print(f'DataFrame shape: {df.shape}')
df.to_parquet('pseudo_label_predictions.parquet', index=False)
print('Saved pseudo_label_predictions.parquet')

# Summary stats
max_probs = preds_np.max(axis=1)
for thresh in [0.5, 0.4, 0.3, 0.2]:
    n = (max_probs >= thresh).sum()
    print(f'  Clips with max_prob >= {thresh}: {n:,} ({100*n/len(max_probs):.1f}%)')